In [1]:
import cupy as cp

In [2]:
import torch

In [3]:
if torch.cuda.is_available():
    device_properties = torch.cuda.get_device_properties(0)
    print(f"Device Name: {device_properties.name}")
    print(f"Total Memory: {device_properties.total_memory / (1024**3):.2f} GB")
    print(f"CUDA Capability: {device_properties.major}.{device_properties.minor}")
else:
    print("CUDA is not available.")

Device Name: NVIDIA GeForce RTX 4090
Total Memory: 23.53 GB
CUDA Capability: 8.9


In [4]:
from minio import Minio
import os
import pandas as pd
import platform
import time

In [5]:
os.name

'posix'

In [6]:
print(platform.system(), platform.release())

Linux 6.8.0-59-generic


In [2]:
os.environ["SSL_CERT_FILE"] = 'all-data/public.crt'

In [3]:
def download_data_without_creds(data):
    client = Minio(
    '94.124.179.195:9000',
    secure=True
    )
    data = client.fget_object(
        'datasets',
        f'data/{data}',
        f'all-data/{data}'
    )

In [6]:
download_data_without_creds('OpenML-har-orig.csv')

In [7]:
download_data_without_creds('OpenML-usps-orig.csv')

In [11]:
# GaMAC Test

In [7]:
from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

In [8]:
def gamac_test(data, target_measures):
    measures = {"BR": Internal.BR, "OS": Internal.OS, "MCR": Internal.MCR, "SYM": Internal.SYM}
    used_measures = [measures[x] for x in target_measures.split(sep=',')]
    df = pd.read_csv(f'devops/comparison/test-data/{data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
        # classes = [int(c[1]) for c in classes]
    else:
        classes = None
    df = df.drop('class', errors='ignore', axis=1)
    print(f'used data: {data}')
    print(f'used measures: {used_measures}')
    result = Gamac(target_measures=tuple(used_measures)).run(table=df, text=None, image=None, classes=classes)
    print(f'optimal.model: {result.model}')
    print(f'clusters: {result.model.labels_}')
    print(f'F1 score: {result.f1_score}')

In [10]:
start = time.time()
gamac_test('OpenML-har-orig-1.csv', 'SYM')
print('Work time:', time.time() - start)

used data: OpenML-har-orig-1.csv
used measures: [<Internal.SYM: ('sym', <function sym at 0x7c606cd37ba0>)>]


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 1.929337501525879s, {'SYM': 3.1830334178980463e-07} ===
=== MEASURES 0.24206328392028809s, {'SYM': 0.0003522895966775774} ===
=== ALGO 8.197380542755127s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 14, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 39.7369601726532s, FailedRun, {'bandwidth': 0.6824906747719257, 'max_iter': 182, 'tol': 9.893421650322088e-05} ===
=== ALGO 3.7530765533447266s, FailedRun, {'name': 'DBSCAN', 'eps': 0.6492547360131439, 'eps_sq': 0.4215317122354971, 'min_samples': 15} ===
=== ALGO 3.940246820449829s, FailedRun, {'min_cluster_size': 15, 'min_samples': 7} ===
=== MEASURES 0.23710227012634277s, {'SYM': 0.0001081383528605122} ===
=== ALGO 90.77414608001709s, SuccessRun, {'threshold': 0.34988546376777646, 'branching_factor': 76, 'n_clusters': 14} ===
=== MEASURES 0.22206854820251465s, {'SYM': 0.00017995791029674517} ===
=== ALGO 0.29221582412719727s, SuccessRun, {'name': 'KMeans', 'n_clusters': 7, 'max_iter': 100, 'tol': 0.0

In [22]:
start = time.time()
gamac_test('OpenML-usps-orig-1.csv', 'SYM')
print('Work time:', time.time() - start)

used data: OpenML-usps-orig-1.csv
used measures: [<Internal.SYM: ('sym', <function sym at 0x7c606cd37ba0>)>]


/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


=== MEASURES 0.25140929222106934s, {'SYM': 3.521961715321675e-07} ===
=== MEASURES 0.24391770362854004s, {'SYM': 0.00011548182376364836} ===
=== ALGO 1.7220203876495361s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 7, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 59.016918897628784s, FailedRun, {'bandwidth': 0.4703932192890266, 'max_iter': 91, 'tol': 2.898634193751156e-05} ===
=== ALGO 12.194046974182129s, FailedRun, {'name': 'DBSCAN', 'eps': 1.1229910037414395, 'eps_sq': 1.2611087944842057, 'min_samples': 9} ===
=== ALGO 3.749473810195923s, FailedRun, {'min_cluster_size': 15, 'min_samples': 11} ===
=== MEASURES 0.2566404342651367s, {'SYM': 5.93169453660373e-05} ===
=== ALGO 143.42006826400757s, SuccessRun, {'threshold': 0.33995949995224284, 'branching_factor': 64, 'n_clusters': 4} ===
=== MEASURES 0.2376081943511963s, {'SYM': 1.4326337353208614e-05} ===
=== ALGO 0.36106276512145996s, SuccessRun, {'name': 'KMeans', 'n_clusters': 2, 'max_iter': 100, 'tol': 0.0

In [ ]:
# cuML test

In [15]:
from cuml.cluster import KMeans
from gamac.estimation.functions import f1

In [20]:
def cuml_kmeans_test(data):
    df = pd.read_csv(f'devops/comparison/test-data/{data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
    else:
        classes = None
    df = df.drop('class', errors='ignore', axis=1)
    print(f'used data: {data}')
    kmeans = KMeans(n_clusters=4, init="k-means++")
    kmeans.fit(df)
    predicted_labels_gpu = cp.array(kmeans.labels_, dtype=cp.int32)
    classes_gpu =  cp.array(classes, dtype=cp.int32)
    f1_score = f1(classes_gpu, predicted_labels_gpu)
    print(f'clusters: {kmeans.labels_}')
    print(f'F1 score: {f1_score}')

In [21]:
start = time.time()
cuml_kmeans_test('OpenML-har-orig-1.csv')
print('Work time:', time.time() - start)

used data: OpenML-har-orig-1.csv
clusters: 0        1
1        1
2        1
3        1
4        1
        ..
10294    0
10295    0
10296    0
10297    0
10298    0
Length: 10299, dtype: int32
F1 score: 0.7091204063083331
Work time: 0.5748333930969238


In [23]:
start = time.time()
cuml_kmeans_test('OpenML-har-orig-1.csv')
print('Work time:', time.time() - start)

used data: OpenML-har-orig-1.csv
clusters: 0        1
1        1
2        1
3        1
4        1
        ..
10294    0
10295    0
10296    0
10297    0
10298    0
Length: 10299, dtype: int32
F1 score: 0.7091204063083331
Work time: 0.6151208877563477


In [24]:
# skicit-learn test

In [25]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [28]:
def sklearn_kmeans_test(data):
    df = pd.read_csv(f'devops/comparison/test-data/{data}')
    if 'class' in df.columns:
        classes = df['class'].tolist()
    else:
        classes = None
    df = df.drop('class', errors='ignore', axis=1)
    print(f'used data: {data}')
    numeric_features = df.select_dtypes(include=['number'])
    categorical_features = df.select_dtypes(exclude=['number'])
    transformer = ColumnTransformer([
        ('numerical', StandardScaler(), numeric_features.columns),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features.columns)
    ])
    transformed_data = transformer.fit_transform(df)
    kmeans = KMeans(n_clusters=4, random_state=42)
    kmeans.fit(transformed_data)
    predicted_labels_gpu = cp.array(kmeans.labels_, dtype=cp.int32)
    classes_gpu =  cp.array(classes, dtype=cp.int32)
    f1_score = f1(classes_gpu, predicted_labels_gpu)
    print(f'clusters: {kmeans.labels_}')
    print(f'F1 score: {f1_score}')

In [29]:
start = time.time()
sklearn_kmeans_test('OpenML-har-orig-1.csv')
print('Work time:', time.time() - start)

used data: OpenML-har-orig-1.csv
clusters: [0 0 0 ... 3 3 3]
F1 score: 0.5297312871363471
Work time: 0.8302676677703857


In [30]:
start = time.time()
sklearn_kmeans_test('OpenML-usps-orig-1.csv')
print('Work time:', time.time() - start)

used data: OpenML-usps-orig-1.csv
clusters: [1 1 2 ... 2 3 0]
F1 score: 0.47587170043935956
Work time: 0.5933609008789062


In [38]:
# auto-sklearn test

In [40]:
from auto_sklearn2 import ClusteringTask, AutoSklearnClustering


ImportError: cannot import name 'ClusteringTask' from 'auto_sklearn2' (/venv/main/lib/python3.12/site-packages/auto_sklearn2/__init__.py)

In [39]:
# https://pypi.org/project/auto-sklearn2/1.0.0/ - обновленный auto-sklearn2 не поддерживает AutoML для задач кластеризаии

In [45]:
# auto-clustering test

In [46]:
from autoclustering import AutoClustering
clustering = AutoClustering(num_samples=50,
                            metric='validity_index',
                            n_jobs=-1,
                            verbose=0)

ImportError: cannot import name '_parse_version' from 'sklearn.utils.fixes' (/venv/main/lib/python3.12/site-packages/sklearn/utils/fixes.py)

In [47]:
# https://pypi.org/project/auto-clustering/ - последняя версия не совместима с другими компонентами актуального образа GaMAC, тестирование невозможно